# Preprocessing of Smart Card data from Santiago, Chile for analysis and ML model implementation

Last reviewed: Thursday 12-05-2026

Times this file has been edited: 6

Hardware specs:

1. *Portable* branch:
    * Machine: MacBook Air (13-inch, 2017)
    * OS: MacOS Monterey v12.7.6
    * CPU: 1.8 GHz Intel Core i5 de dos núcleos
    * RAM: 8 GB 1600 MHz DDR3
    * Graphics: Intel HD Graphics 6000 1536 MB

2. *House* branch:
    * OS: Windows 10 Home 64-bit (10.0, Build 19045)
    * Processor: Intel(R) Core(TM) i7-8700K CPU @ 3.70GHz (12 CPUs), ~3.7GHz
    * Memory: 12288MB RAM
    * GPU: NVIDIA GeForce RTX 3060 12115 MB

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import duckdb
from pathlib import Path

## Script 01: Conversión de datos a formato Parquet

In [6]:
"""
Script 01: Conversión de datos raw a formato Parquet
Sin transformaciones — solo lectura y escritura eficiente.
Datasets: Trips, Stages, GPS
"""

# ──── Configuración de DuckDB para manejo eficiente de grandes archivos ──────

def crear_conexion() -> duckdb.DuckDBPyConnection:
    """
    Crea una conexión DuckDB con límites de recursos seguros.
    Ajusta memory_limit según tu RAM disponible:
      - 8 GB RAM  → usa '4GB'
      - 16 GB RAM → usa '8GB'
      - 32 GB RAM → usa '16GB'
    """
    con = duckdb.connect()
    con.execute("SET memory_limit = '6GB'")       # ajusta según tu RAM
    con.execute("SET threads = 6")                 # ajusta según tus núcleos
    con.execute("SET temp_directory = 'D:/temp_duckdb'")  # disco para overflow
    return con

# ── Rutas base ───────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
RAW_DIR   = BASE_DIR / "raw"
PROC_DIR  = BASE_DIR / "parquets"


# ── Funciones auxiliares ─────────────────────────────────────────────────────

def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def archivos_de_carpeta(carpeta: Path, extensiones: list[str]) -> list[Path]:
    archivos = []
    for ext in extensiones:
        archivos.extend(sorted(carpeta.glob(f"*{ext}")))
    return archivos


def reportar_compresion(archivos_raw: list[Path], parquet_path: Path):
    size_raw     = sum(f.stat().st_size for f in archivos_raw) / 1e6
    size_parquet = parquet_path.stat().st_size / 1e6
    ratio        = size_raw / size_parquet if size_parquet > 0 else 0
    print(f"  Raw:     {size_raw:.1f} MB")
    print(f"  Parquet: {size_parquet:.1f} MB")
    print(f"  Ratio:   {ratio:.1f}x")


# ── Trips ────────────────────────────────────────────────────────────────────

def convertir_trips(anio: str):
    carpeta_in  = RAW_DIR  / "Trips" / anio
    carpeta_out = PROC_DIR / "Trips" / anio
    asegurar_carpeta(carpeta_out)

    archivos = archivos_de_carpeta(carpeta_in, [".csv", ".viajes"])
    if not archivos:
        print(f"  [AVISO] No se encontraron archivos en {carpeta_in}")
        return

    salida = carpeta_out / f"trips_{anio}_raw.parquet"
    print(f"\n→ Trips {anio}: {len(archivos)} archivo(s)")

    # Construimos la query como UNION ALL de todos los archivos
    # DuckDB escribe directo a Parquet sin pasar por memoria Python
    union_parts = []
    for archivo in archivos:
        fecha = archivo.name.split(".")[0]
        # Usamos barras normales — DuckDB en Windows acepta ambas
        ruta = str(archivo).replace("\\", "/")
        union_parts.append(f"""
            SELECT
                '{fecha}' AS fecha,
                *
            FROM read_csv(
                '{ruta}',
                delim         = '|',
                header        = true,
                encoding      = 'cp1252',
                all_varchar   = true,
                null_padding  = true,
                ignore_errors = true
            )
        """)

    query_union = "\nUNION ALL\n".join(union_parts)
    salida_str  = str(salida).replace("\\", "/")

    con = crear_conexion()
    con.execute(f"""
        COPY (
            {query_union}
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """)

    # Contar filas sin cargar en memoria
    n_filas = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    con.close()

    print(f"  ✓ {salida.name} — {n_filas:,} filas totales")
    reportar_compresion(archivos, salida)


# ── Stages ───────────────────────────────────────────────────────────────────

def convertir_stages(anio: str):
    carpeta_in  = RAW_DIR  / "Stages" / anio
    carpeta_out = PROC_DIR / "Stages" / anio
    asegurar_carpeta(carpeta_out)

    archivos = archivos_de_carpeta(carpeta_in, [".csv", ".etapas"])
    if not archivos:
        print(f"  [AVISO] No se encontraron archivos en {carpeta_in}")
        return

    salida = carpeta_out / f"stages_{anio}_raw.parquet"
    print(f"\n→ Stages {anio}: {len(archivos)} archivo(s)")

    union_parts = []
    for archivo in archivos:
        fecha = archivo.name.split(".")[0]
        ruta  = str(archivo).replace("\\", "/")
        union_parts.append(f"""
            SELECT
                '{fecha}' AS fecha,
                *
            FROM read_csv(
                '{ruta}',
                delim         = '|',
                header        = true,
                encoding      = 'cp1252',
                all_varchar   = true,
                null_padding  = true,
                ignore_errors = true
            )
        """)

    query_union = "\nUNION ALL\n".join(union_parts)
    salida_str  = str(salida).replace("\\", "/")

    con = crear_conexion()
    con.execute(f"""
        COPY (
            {query_union}
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """)

    n_filas = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    con.close()

    print(f"  ✓ {salida.name} — {n_filas:,} filas totales")
    reportar_compresion(archivos, salida)


# ── GPS ──────────────────────────────────────────────────────────────────────

def convertir_gps(anio: str):
    carpeta_in  = RAW_DIR  / "GPS" / anio
    carpeta_out = PROC_DIR / "GPS" / anio
    asegurar_carpeta(carpeta_out)

    archivos = archivos_de_carpeta(carpeta_in, [".gps", ".csv"])
    if not archivos:
        print(f"  [AVISO] No se encontraron archivos en {carpeta_in}")
        return

    salida = carpeta_out / f"gps_{anio}_raw.parquet"
    print(f"\n→ GPS {anio}: {len(archivos)} archivo(s)")

    union_parts = []
    for archivo in archivos:
        fecha = archivo.name.split(".")[0]
        ruta  = str(archivo).replace("\\", "/")
        union_parts.append(f"""
            SELECT
                '{fecha}' AS fecha_archivo,
                *
            FROM read_csv(
                '{ruta}',
                delim         = ';',
                header        = false,
                encoding      = 'cp1252',
                all_varchar   = true,
                ignore_errors = true
            )
        """)

    query_union = "\nUNION ALL\n".join(union_parts)
    salida_str  = str(salida).replace("\\", "/")

    con = crear_conexion()
    con.execute(f"""
        COPY (
            {query_union}
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """)

    n_filas = con.execute(f"""
        SELECT COUNT(*) FROM read_parquet('{salida_str}')
    """).fetchone()[0]
    con.close()

    print(f"  ✓ {salida.name} — {n_filas:,} filas totales")
    reportar_compresion(archivos, salida)

In [7]:
# ── Ejecución ────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 60)
    print("CONVERSIÓN RAW → PARQUET (sin transformaciones)")
    print("=" * 60)

    for anio in ["2024", "2025"]: # Agregar otros años según disponibilidad y necesidad
        convertir_trips(anio)

    for anio in ["2025"]:
        convertir_stages(anio)

    for anio in ["2018", "2022"]:
        convertir_gps(anio)

    print("\n" + "=" * 60)
    print("✓ Conversión completada")
    print("=" * 60)

CONVERSIÓN RAW → PARQUET (sin transformaciones)

→ Trips 2024: 9 archivo(s)
  ✓ trips_2024_raw.parquet — 24,430,419 filas totales
  Raw:     9974.8 MB
  Parquet: 1727.0 MB
  Ratio:   5.8x

→ Trips 2025: 7 archivo(s)
  ✓ trips_2025_raw.parquet — 21,313,043 filas totales
  Raw:     8778.0 MB
  Parquet: 1535.9 MB
  Ratio:   5.7x

→ Stages 2025: 7 archivo(s)
  ✓ stages_2025_raw.parquet — 27,949,622 filas totales
  Raw:     8249.0 MB
  Parquet: 1340.6 MB
  Ratio:   6.2x

→ GPS 2018: 7 archivo(s)
  ✓ gps_2018_raw.parquet — 77,761,905 filas totales
  Raw:     6008.3 MB
  Parquet: 1172.1 MB
  Ratio:   5.1x

→ GPS 2022: 14 archivo(s)
  ✓ gps_2022_raw.parquet — 172,111,519 filas totales
  Raw:     12123.8 MB
  Parquet: 1304.5 MB
  Ratio:   9.3x

✓ Conversión completada


## Script 02: Agregación del clima con API externa

In [ ]:
"""
Script 02: Descarga de datos climáticos históricos desde Open-Meteo
para los períodos cubiertos por los datasets de viajes y etapas.

Estrategia:
- Se descarga una vez por período y se guarda como Parquet independiente.
- El join con Trips/Stages se realiza en tiempo de consulta con DuckDB.
- No se duplican datos climáticos en cada dataset.
- Resolución: horaria, coordenadas centradas en Santiago.
"""

import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
from pathlib import Path

# ── Rutas base ───────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
PROC_DIR  = BASE_DIR / "parquets"
CLIMA_DIR = PROC_DIR / "02_enriched" / "clima"


# ── Coordenadas de Santiago ──────────────────────────────────────────────────
# Centro geográfico aproximado del Gran Santiago
# Suficientemente preciso para clima — la variación intra-ciudad es mínima
# para temperatura y precipitación a escala horaria
SANTIAGO_LAT =  -33.4489
SANTIAGO_LON =  -70.6693
TIMEZONE     = "America/Santiago"


# ── Variables a descargar ────────────────────────────────────────────────────
VARIABLES_HORARIAS = [
    "temperature_2m",       # temperatura en °C
    "precipitation",        # precipitación total mm/h
    "rain",                 # lluvia (excluye nieve) mm/h
    "weather_code",         # código WMO de condición meteorológica
    "wind_speed_10m",       # velocidad del viento km/h
    "relative_humidity_2m", # humedad relativa %
]


# ── Períodos a descargar ─────────────────────────────────────────────────────
# Agregar aquí todos los períodos que tengas en tus datasets
# formato: (nombre_periodo, fecha_inicio, fecha_fin)
PERIODOS = [
    ("2019_mayo",      "2019-05-19", "2019-05-19"),
    ("2024_noviembre", "2024-11-09", "2024-11-17"),
    ("2025_abril",     "2025-04-21", "2025-04-27"),
]


# ── Cliente Open-Meteo con caché y reintentos ────────────────────────────────
def crear_cliente():
    """
    Crea un cliente Open-Meteo con:
    - Caché en disco: evita re-descargar si el script se ejecuta más de una vez
    - Reintentos automáticos: maneja cortes de red transitorios
    """
    cache_session = requests_cache.CachedSession(
        str(BASE_DIR / "processed" / "02_enriched" / ".cache_openmeteo"),
        expire_after = -1  # caché permanente — datos históricos no cambian
    )
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    return openmeteo_requests.Client(session=retry_session)


# ── Descarga y procesamiento ─────────────────────────────────────────────────

def descargar_clima_periodo(
    cliente,
    nombre: str,
    fecha_inicio: str,
    fecha_fin: str
) -> pd.DataFrame:
    """
    Descarga datos horarios de Open-Meteo para un período dado
    y devuelve un DataFrame con una fila por hora.
    """
    print(f"  Descargando {nombre} ({fecha_inicio} → {fecha_fin})...")

    params = {
        "latitude":   SANTIAGO_LAT,
        "longitude":  SANTIAGO_LON,
        "start_date": fecha_inicio,
        "end_date":   fecha_fin,
        "hourly":     VARIABLES_HORARIAS,
        "timezone":   TIMEZONE,
    }

    respuestas = cliente.weather_api(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params
    )
    resp = respuestas[0]  # una sola ubicación → primer elemento

    # Extraer datos horarios
    horario = resp.Hourly()

    df = pd.DataFrame({
        "timestamp": pd.date_range(
            start = pd.Timestamp(horario.Time(), unit="s", tz=TIMEZONE),
            end   = pd.Timestamp(horario.TimeEnd(), unit="s", tz=TIMEZONE),
            freq  = pd.Timedelta(seconds=horario.Interval()),
            inclusive = "left"
        ),
        "temperature_2m":       horario.Variables(0).ValuesAsNumpy(),
        "precipitation":        horario.Variables(1).ValuesAsNumpy(),
        "rain":                 horario.Variables(2).ValuesAsNumpy(),
        "weather_code":         horario.Variables(3).ValuesAsNumpy(),
        "wind_speed_10m":       horario.Variables(4).ValuesAsNumpy(),
        "relative_humidity_2m": horario.Variables(5).ValuesAsNumpy(),
    })

    # Columnas derivadas útiles para el modelo
    df["fecha"]         = df["timestamp"].dt.date.astype(str)
    df["hora"]          = df["timestamp"].dt.hour
    df["mediahora"]     = (df["hora"] * 2 + (df["timestamp"].dt.minute >= 30).astype(int))
    df["llueve"]        = (df["rain"] > 0.1).astype(int)          # binario: llueve o no
    df["lluvia_intensa"]= (df["rain"] > 2.0).astype(int)          # > 2mm/h = lluvia intensa
    df["periodo_clima"] = nombre

    # Clasificación de condición meteorológica según código WMO
    # https://open-meteo.com/en/docs — sección Weather variable descriptions
    df["condicion"] = df["weather_code"].map(clasificar_wmo).fillna("desconocido")

    return df


def clasificar_wmo(codigo: float) -> str:
    """
    Clasifica el código WMO en categorías legibles.
    Referencia: https://open-meteo.com/en/docs (WMO Weather interpretation codes)
    """
    if pd.isna(codigo):
        return "desconocido"
    c = int(codigo)
    if c == 0:                    return "despejado"
    elif c in (1, 2, 3):          return "nublado"
    elif c in (45, 48):           return "niebla"
    elif c in (51, 53, 55):       return "llovizna"
    elif c in (61, 63, 65):       return "lluvia"
    elif c in (71, 73, 75, 77):   return "nieve"
    elif c in (80, 81, 82):       return "chubascos"
    elif c in (95, 96, 99):       return "tormenta"
    else:                         return "otro"

DESCARGA DE DATOS CLIMÁTICOS — Open-Meteo
Ubicación: Santiago (-33.4489, -70.6693)
  Descargando 2019_mayo (2019-05-19 → 2019-05-19)...
  ✓ clima_2019_mayo.parquet — 24 filas (0 horas con lluvia)
  Descargando 2024_noviembre (2024-11-09 → 2024-11-17)...
  ✓ clima_2024_noviembre.parquet — 216 filas (1 horas con lluvia)
  Descargando 2025_abril (2025-04-21 → 2025-04-27)...
  ✓ clima_2025_abril.parquet — 168 filas (0 horas con lluvia)

  ✓ Archivo unificado: clima_todos_periodos.parquet — 408 filas totales

✓ Descarga completada


In [ ]:
# ── Ejecución ────────────────────────────────────────────────────────────────

def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


if __name__ == "__main__":

    asegurar_carpeta(CLIMA_DIR)

    print("=" * 60)
    print("DESCARGA DE DATOS CLIMÁTICOS — Open-Meteo")
    print(f"Ubicación: Santiago ({SANTIAGO_LAT}, {SANTIAGO_LON})")
    print("=" * 60)

    cliente = crear_cliente()
    dfs = []

    for nombre, fecha_inicio, fecha_fin in PERIODOS:
        salida = CLIMA_DIR / f"clima_{nombre}.parquet"

        # Si ya existe el archivo, no re-descarga
        if salida.exists():
            print(f"  [SKIP] {salida.name} ya existe — usa caché o borra para re-descargar")
            continue

        df = descargar_clima_periodo(cliente, nombre, fecha_inicio, fecha_fin)
        df.to_parquet(salida, compression="zstd", index=False)

        print(f"  ✓ {salida.name} — {len(df):,} filas ({df['llueve'].sum()} horas con lluvia)")
        dfs.append(df)

    # Unificar todos los períodos en un solo archivo de referencia
    salida_total = CLIMA_DIR / "clima_todos_periodos.parquet"
    if not salida_total.exists():
        todos = pd.concat(
            [pd.read_parquet(f) for f in sorted(CLIMA_DIR.glob("clima_*.parquet"))
             if "todos" not in f.name],
            ignore_index=True
        )
        todos.to_parquet(salida_total, compression="zstd", index=False)
        print(f"\n  ✓ Archivo unificado: {salida_total.name} — {len(todos):,} filas totales")

    print("\n" + "=" * 60)
    print("✓ Descarga completada")
    print("=" * 60)

## Script 03: Agregación de rutas y horarios mediante GTFS del sistema RED (vigente desde el 21 de marzo de 2026)

In [1]:
import pandas as pd
from pathlib import Path

gtfs = Path(r'D:\GitHub\tesis_magister_route_choice_modelling\data\raw\GTFS (2026-03-21)')

for archivo in ['stops', 'routes', 'trips', 'stop_times', 'frequencies']:
    ruta = gtfs / f'{archivo}.txt'
    if ruta.exists():
        df = pd.read_csv(ruta, nrows=3, encoding='utf-8', sep=',')
        print(f'=== {archivo}.txt ===')
        print(f'Columnas: {list(df.columns)}')
        print(df.to_string())
        print()

=== stops.txt ===
Columnas: ['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon', 'stop_url', 'wheelchair_boarding', 'location_type', 'parent_station', 'level_id']
  stop_id  stop_code                               stop_name   stop_lat   stop_lon  stop_url  wheelchair_boarding  location_type  parent_station  level_id
0  PD1641        NaN             PD1641-Parada 7 / (M) Macul -33.509178 -70.589577       NaN                    0            NaN             NaN       NaN
1   PE158        NaN  PE158-Parada / Colegio José A. Lecaros -33.514682 -70.592525       NaN                    0            NaN             NaN       NaN
2   PE159        NaN      PE159-Parada / Fiscalía La Florida -33.516556 -70.594188       NaN                    0            NaN             NaN       NaN

=== routes.txt ===
Columnas: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color']
   route_id agency_id  route_short_na

In [2]:
import pandas as pd
from pathlib import Path

gtfs = Path(r'D:\GitHub\tesis_magister_route_choice_modelling\data\raw\GTFS (2026-03-21)')

for archivo in ['agency', 'calendar', 'calendar_dates', 'feed_info', 'shapes', 'levels', 'pathways']:
    ruta = gtfs / f'{archivo}.txt'
    if ruta.exists():
        df = pd.read_csv(ruta, nrows=3, encoding='utf-8', sep=',')
        print(f'=== {archivo}.txt ===')
        print(f'Columnas: {list(df.columns)}')
        print(df.to_string())
        print()
    else:
        print(f'=== {archivo}.txt === [NO EXISTE]')
        print()

=== agency.txt ===
Columnas: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_phone', 'agency_fare_url', 'agency_email', 'agency_lang']
  agency_id                     agency_name                   agency_url   agency_timezone  agency_phone                         agency_fare_url  agency_email  agency_lang
0        RM  Red Metropolitana de Movilidad            http://www.red.cl  America/Santiago           NaN  https://www.red.cl/tarifas-y-recargas/           NaN          NaN
1        MT             EFE Trenes de Chile            http://www.efe.cl  America/Santiago           NaN  https://www.red.cl/tarifas-y-recargas/           NaN          NaN
2       BAA  Bus de Acercamiento Aeropuerto  http://www.nuevopudahuel.cl  America/Santiago           NaN  https://www.red.cl/tarifas-y-recargas/           NaN          NaN

=== calendar.txt ===
Columnas: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
  se

In [4]:
import duckdb

con = duckdb.connect()

print('=== Paraderos en Stages 2025 ===')
con.sql('''
    SELECT DISTINCT parada_subida, parada_bajada
    FROM read_parquet(
        \"D:/GitHub/tesis_magister_route_choice_modelling/data/parquets/01_raw/Stages/2025/stages_2025_raw.parquet\"
    )
    WHERE parada_subida IS NOT NULL
    LIMIT 20
''').show()

print()
print('=== Paraderos en Trips 2024 ===')
con.sql('''
    SELECT DISTINCT paradero_subida_1, paradero_bajada_1
    FROM read_parquet(
        \"D:/GitHub/tesis_magister_route_choice_modelling/data/parquets/01_raw/Trips/2024/trips_2024_raw.parquet\"
    )
    WHERE paradero_subida_1 IS NOT NULL
      AND paradero_subida_1 != \'-\'
    LIMIT 20
''').show()

con.close()

=== Paraderos en Stages 2025 ===
┌──────────────────────┬──────────────────┐
│    parada_subida     │  parada_bajada   │
│       varchar        │     varchar      │
├──────────────────────┼──────────────────┤
│ T-20-188-SN-54       │ -                │
│ T-34-313-SN-15       │ T-33-89-PO-5     │
│ UNIVERSIDAD DE CHILE │ LO OVALLE        │
│ PEDRO DE VALDIVIA    │ TOBALABA         │
│ T-27-228-NS-30       │ T-27-228-NS-35   │
│ T-6-47-SN-10         │ L-6-5-45-OP      │
│ T-17-139-OP-20       │ E-17-140-OP-65   │
│ T-20-53-PO-85        │ -                │
│ T-20-59-OP-40        │ T-8-64-OP-35     │
│ L-13-14-141-PO       │ T-12-88-SN-15    │
│ T-32-316-OP-10       │ -                │
│ UNIVERSIDAD DE CHILE │ EL PARRON        │
│ T-20-405-SN-5        │ -                │
│ PARQUE BUSTAMANTE    │ MIRADOR          │
│ T-12-55-NS-15        │ -                │
│ T-30-244-SN-10       │ T-27-230-NS-65   │
│ T-18-159-SN-10       │ T-14-123-SN-45   │
│ BLANQUEADO           │ LOS LIBERTADORES │

In [2]:
import pandas as pd
from pathlib import Path

gtfs = Path(r'D:\GitHub\tesis_magister_route_choice_modelling\data\raw\GTFS (2026-03-21)')

# Verificar si stop_code tiene valores no nulos
stops = pd.read_csv(gtfs / 'stops.txt', encoding='utf-8')
print('=== stop_code no nulos ===')
print(f'Total stops: {len(stops)}')
print(f'stop_code no nulos: {stops["stop_code"].notna().sum()}')
print()
print('=== Muestra de stops con stop_code no nulo ===')
print(stops[stops['stop_code'].notna()].head(10).to_string())
print()

# Ver si stop_name contiene el código interno del bip
print('=== Muestra stop_name completa ===')
print(stops[['stop_id', 'stop_code', 'stop_name']].head(20).to_string())
print()

# Buscar si algún stop_id o stop_name contiene el formato T-XX o L-XX
import re
patron = re.compile(r'^[TLE]-\d+-\d+')
coincidencias = stops[
    stops['stop_id'].str.match(patron, na=False) |
    stops['stop_name'].str.contains(patron, na=False)
]
print(f'=== Stops con formato tipo bip! (T-, L-, E-): {len(coincidencias)} ===')
print(coincidencias.head(10).to_string())

=== stop_code no nulos ===
Total stops: 18364
stop_code no nulos: 28

=== Muestra de stops con stop_code no nulo ===
      stop_id stop_code                              stop_name  stop_lat  stop_lon  stop_url  wheelchair_boarding  location_type parent_station level_id
18330  PT0201    PT0101              Estación Central (Anden1) -33.45115 -70.67880       NaN                    1            NaN            NaN      NaN
18331  PT0202    PT0102          Estación Lo Valledor (Anden1) -33.47800 -70.68060       NaN                    1            NaN            NaN      NaN
18332  PT0203    PT0103  Estación Pedro Aguirre Cerda (Anden1) -33.49310 -70.68180       NaN                    1            NaN            NaN      NaN
18333  PT0204    PT0104            Estación Lo Espejo (Anden1) -33.51370 -70.68570       NaN                    1            NaN            NaN      NaN
18334  PT0205    PT0105            Estación Lo Blanco (Anden1) -33.57280 -70.69829       NaN                    1     

In [8]:
import pandas as pd
import duckdb
from pyproj import Transformer

transformer = Transformer.from_crs('EPSG:32719', 'EPSG:4326', always_xy=True)

con = duckdb.connect()
df = con.sql('''
    SELECT
        parada_subida,
        x_subida,
        y_subida
    FROM read_parquet(
        'D:/GitHub/tesis_magister_route_choice_modelling/data/parquets/01_raw/Stages/2025/stages_2025_raw.parquet'
    )
    WHERE x_subida IS NOT NULL
      AND y_subida IS NOT NULL
      AND x_subida != cast(-1 as varchar)
    LIMIT 10
''').df()
con.close()

df['x_float'] = pd.to_numeric(df['x_subida'], errors='coerce')
df['y_float'] = pd.to_numeric(df['y_subida'], errors='coerce')
df = df.dropna(subset=['x_float', 'y_float'])

lon, lat = transformer.transform(df['x_float'].values, df['y_float'].values)
df['lon_wgs84'] = lon
df['lat_wgs84'] = lat

print('=== Conversión UTM → WGS84 ===')
print(df[['parada_subida', 'x_subida', 'y_subida', 'lat_wgs84', 'lon_wgs84']].to_string())
print()
print('Rango latitudes:', round(df['lat_wgs84'].min(), 4), 'a', round(df['lat_wgs84'].max(), 4))
print('Rango longitudes:', round(df['lon_wgs84'].min(), 4), 'a', round(df['lon_wgs84'].max(), 4))
print()
print('Esperado para Santiago: lat ~ -33.3 a -33.6 / lon ~ -70.4 a -70.8')

=== Conversión UTM → WGS84 ===
              parada_subida x_subida y_subida  lat_wgs84  lon_wgs84
0              T-4-19-SN-40   347180  6301636 -33.413750 -70.643536
1              E-4-19-SN-55   347200  6302473 -33.406206 -70.643179
2              L-4-12-20-PO   346563  6303315 -33.398524 -70.649883
3             LOS DOMINICOS   356329  6302427 -33.407882 -70.545047
4          ESTACION CENTRAL   343933  6297455 -33.450978 -70.679170
5            PLAZA DE ARMAS   346372  6299031 -33.437121 -70.652668
6  BELLAVISTA DE LA FLORIDA   351430  6289929 -33.519899 -70.599783
7               MONTE TABOR   337829  6293884 -33.482265 -70.745464
8            T-14-131-PO-10   348577  6299461 -33.433558 -70.628883
9             T-15-135-PO-5   351342  6302517 -33.406391 -70.598645

Rango latitudes: -33.5199 a -33.3985
Rango longitudes: -70.7455 a -70.545

Esperado para Santiago: lat ~ -33.3 a -33.6 / lon ~ -70.4 a -70.8


In [17]:
"""
Script 03: Integración de datos GTFS al pipeline de enriquecimiento.

Construye tres tablas de referencia:
  A) Diccionario bip! → GTFS stop_id  (join espacial por proximidad)
  B) Tabla de rutas por paradero       (qué líneas pasan por cada stop)
  C) Tabla de frecuencias por período  (headway en cada franja horaria)

Estas tablas se guardan como Parquet y se usan en tiempo de consulta.
No se modifican los datasets raw — solo se agregan llaves de join.
"""

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from pyproj import Transformer
from sklearn.neighbors import BallTree

# ── Rutas base ───────────────────────────────────────────────────────────────
BASE_DIR   = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
PROC_DIR   = BASE_DIR / "parquets"
GTFS_DIR   = BASE_DIR / "raw" / "GTFS (2026-03-21)"
GTFS_OUT   = PROC_DIR / "02_enriched" / "gtfs"

STAGES_2025 = PROC_DIR / "01_raw" / "Stages" / "2025" / "stages_2025_raw.parquet"

# Transformador UTM zona 19S → WGS84
TRANSFORMER = Transformer.from_crs("EPSG:32719", "EPSG:4326", always_xy=True)

# Radio máximo de matching en metros
# Si el vecino más cercano está a más de este radio, se descarta el match
RADIO_MAX_METROS = 50 # Editar según la precisión esperada de los paraderos en Stages


def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


# ── Tabla A: Diccionario bip! → GTFS stop_id ────────────────────────────────

def construir_diccionario_paraderos() -> pd.DataFrame:
    """
    Para cada paradero único en los datos de etapas, encuentra el stop_id
    GTFS más cercano usando un BallTree sobre coordenadas WGS84.

    Devuelve un DataFrame con columnas:
        parada_bip, x_utm, y_utm, lat, lon,
        gtfs_stop_id, gtfs_stop_name, distancia_metros
    """
    print("\n→ Construyendo diccionario bip! → GTFS...")

    # 1. Obtener todos los paraderos únicos con coordenadas desde stages
    con = duckdb.connect()
    df_bip = con.sql(f"""
        SELECT
            parada_subida  AS parada_bip,
            AVG(TRY_CAST(x_subida AS DOUBLE)) AS x_utm,
            AVG(TRY_CAST(y_subida AS DOUBLE)) AS y_utm
        FROM read_parquet('{str(STAGES_2025).replace(chr(92), "/")}')
        WHERE x_subida IS NOT NULL
          AND y_subida IS NOT NULL
          AND TRY_CAST(x_subida AS DOUBLE) IS NOT NULL
          AND TRY_CAST(y_subida AS DOUBLE) IS NOT NULL
          AND TRY_CAST(x_subida AS DOUBLE) > 0
        GROUP BY parada_subida
    """).df()
    con.close()

    print(f"  Paraderos únicos en stages: {len(df_bip):,}")

    # 2. Convertir UTM → WGS84
    lon, lat = TRANSFORMER.transform(df_bip["x_utm"].values, df_bip["y_utm"].values)
    df_bip["lat"] = lat
    df_bip["lon"] = lon

    # 3. Cargar stops del GTFS
    # Solo stops de tipo paradero/estación (location_type NaN o 0)
    # Excluir entradas/salidas de Metro (location_type 2, 3, 4)
    stops = pd.read_csv(GTFS_DIR / "stops.txt", encoding="utf-8")
    stops_filtrados = stops[
        stops["location_type"].isna() | stops["location_type"].isin([0, 1])
    ].copy()
    stops_filtrados = stops_filtrados.dropna(subset=["stop_lat", "stop_lon"])

    print(f"  Stops GTFS disponibles para matching: {len(stops_filtrados):,}")

    # 4. Construir BallTree sobre coordenadas GTFS
    # BallTree trabaja en radianes para distancias esféricas
    coords_gtfs = np.radians(
        stops_filtrados[["stop_lat", "stop_lon"]].values
    )
    tree = BallTree(coords_gtfs, metric="haversine")

    # 5. Buscar vecino más cercano para cada paradero bip!
    coords_bip = np.radians(df_bip[["lat", "lon"]].values)
    distancias, indices = tree.query(coords_bip, k=1)

    # Convertir distancia haversine (radianes) a metros
    RADIO_TIERRA = 6_371_000
    distancias_metros = distancias.flatten() * RADIO_TIERRA

    # 6. Asignar resultado y filtrar por radio máximo
    stops_reset = stops_filtrados.reset_index(drop=True)
    df_bip["gtfs_stop_id"]   = stops_reset.loc[indices.flatten(), "stop_id"].values
    df_bip["gtfs_stop_name"] = stops_reset.loc[indices.flatten(), "stop_name"].values
    df_bip["distancia_metros"] = distancias_metros

    # Clasificar cada paradero según calidad del match
    def clasificar_match(distancia: float) -> str:
        if distancia <= 50:   return "exacto"
        elif distancia <= 100: return "cercano"
        else:                  return "sin_match"

    df_bip["calidad_match"] = df_bip["distancia_metros"].apply(clasificar_match)

    # Limpiar stop_id para los sin match
    df_bip.loc[
        df_bip["calidad_match"] == "sin_match", "gtfs_stop_id"
    ] = None
    df_bip.loc[
        df_bip["calidad_match"] == "sin_match", "gtfs_stop_name"
    ] = None

    # Estadísticas de cobertura
    exactos  = (df_bip["calidad_match"] == "exacto").sum()
    cercanos = (df_bip["calidad_match"] == "cercano").sum()
    sin      = (df_bip["calidad_match"] == "sin_match").sum()
    total    = len(df_bip)

    print(f"  Match exacto   (≤ 50m):  {exactos:,}  ({exactos/total*100:.1f}%)")
    print(f"  Match cercano  (≤ 100m): {cercanos:,}  ({cercanos/total*100:.1f}%)")
    print(f"  Sin match      (> 100m): {sin:,}   ({sin/total*100:.1f}%)")
    print(f"  Cobertura total:         {exactos+cercanos:,} ({(exactos+cercanos)/total*100:.1f}%)")

    if sin > 0:
        print(f"\n  Paraderos sin match confirmado ({sin}):")
        sin_df = df_bip[df_bip["calidad_match"] == "sin_match"][
            ["parada_bip", "lat", "lon", "distancia_metros"]
        ]
        print(sin_df.to_string())

    return df_bip


# ── Tabla B: Rutas por paradero ──────────────────────────────────────────────

def construir_rutas_por_paradero() -> pd.DataFrame:
    """
    Para cada stop_id GTFS, determina qué rutas (líneas) pasan por él
    y en qué dirección.

    Devuelve un DataFrame con columnas:
        stop_id, route_id, route_short_name, direction_id,
        agency_id, n_trips
    """
    print("\n→ Construyendo tabla de rutas por paradero...")

    stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt", encoding="utf-8",
                             usecols=["trip_id", "stop_id"])
    trips      = pd.read_csv(GTFS_DIR / "trips.txt", encoding="utf-8",
                             usecols=["trip_id", "route_id", "direction_id", "service_id"])
    routes     = pd.read_csv(GTFS_DIR / "routes.txt", encoding="utf-8",
                             usecols=["route_id", "route_short_name", "agency_id"])

    # Join: stop_times → trips → routes
    df = (stop_times
          .merge(trips,  on="trip_id",  how="left")
          .merge(routes, on="route_id", how="left"))

    # Agrupar: por cada (stop, ruta, dirección) contar cuántos trips la usan
    rutas_por_stop = (df
        .groupby(["stop_id", "route_id", "route_short_name",
                  "direction_id", "agency_id"])
        .agg(n_trips=("trip_id", "nunique"))
        .reset_index())

    print(f"  Combinaciones stop × ruta × dirección: {len(rutas_por_stop):,}")

    return rutas_por_stop


# ── Tabla C: Frecuencias por ruta y período ──────────────────────────────────

def construir_frecuencias() -> pd.DataFrame:
    """
    Para cada trip_id, calcula el headway promedio por franja horaria
    y lo asocia a su route_id y service_id.

    El headway (tiempo entre buses) es el atributo de servicio más
    directamente ligado al tiempo de espera del usuario.

    Devuelve un DataFrame con columnas:
        route_id, service_id, direction_id,
        start_time, end_time,
        headway_secs, headway_min
    """
    print("\n→ Construyendo tabla de frecuencias por ruta...")

    frequencies = pd.read_csv(GTFS_DIR / "frequencies.txt", encoding="utf-8")
    trips       = pd.read_csv(GTFS_DIR / "trips.txt", encoding="utf-8",
                              usecols=["trip_id", "route_id",
                                       "direction_id", "service_id"])

    df = frequencies.merge(trips, on="trip_id", how="left")
    df["headway_min"] = df["headway_secs"] / 60

    # Headway promedio por ruta, dirección, servicio y franja
    freq_agrupado = (df
        .groupby(["route_id", "service_id", "direction_id",
                  "start_time", "end_time"])
        .agg(
            headway_secs = ("headway_secs", "mean"),
            headway_min  = ("headway_min",  "mean"),
            n_trips      = ("trip_id",      "nunique")
        )
        .reset_index())

    print(f"  Combinaciones ruta × servicio × franja: {len(freq_agrupado):,}")
    print(f"  Headway promedio global: {freq_agrupado['headway_min'].mean():.1f} min")

    return freq_agrupado

In [18]:
# ── Ejecución ────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    asegurar_carpeta(GTFS_OUT)

    print("=" * 60)
    print("ENRIQUECIMIENTO GTFS")
    print("=" * 60)

    # Tabla A — diccionario de paraderos
    df_dict = construir_diccionario_paraderos()
    salida_dict = GTFS_OUT / "diccionario_paraderos.parquet"
    df_dict.to_parquet(salida_dict, compression="zstd", index=False)
    print(f"\n  ✓ Guardado: {salida_dict.name}")

    # Tabla B — rutas por paradero
    df_rutas = construir_rutas_por_paradero()
    salida_rutas = GTFS_OUT / "rutas_por_paradero.parquet"
    df_rutas.to_parquet(salida_rutas, compression="zstd", index=False)
    print(f"  ✓ Guardado: {salida_rutas.name}")

    # Tabla C — frecuencias
    df_freq = construir_frecuencias()
    salida_freq = GTFS_OUT / "frecuencias_por_ruta.parquet"
    df_freq.to_parquet(salida_freq, compression="zstd", index=False)
    print(f"  ✓ Guardado: {salida_freq.name}")

    print("\n" + "=" * 60)
    print("✓ Enriquecimiento GTFS completado")
    print("=" * 60)
    print("""
Tablas generadas:
  diccionario_paraderos.parquet  → llave bip! ↔ GTFS stop_id
  rutas_por_paradero.parquet     → qué líneas pasan por cada stop
  frecuencias_por_ruta.parquet   → headway por ruta y franja horaria

Ejemplo de uso en DuckDB:
  SELECT
      e.parada_subida,
      d.gtfs_stop_id,
      d.distancia_metros,
      r.route_short_name,
      f.headway_min
  FROM stages e
  LEFT JOIN diccionario_paraderos d ON e.parada_subida = d.parada_bip
  LEFT JOIN rutas_por_paradero    r ON d.gtfs_stop_id  = r.stop_id
  LEFT JOIN frecuencias_por_ruta  f ON r.route_id      = f.route_id
""")

ENRIQUECIMIENTO GTFS

→ Construyendo diccionario bip! → GTFS...
  Paraderos únicos en stages: 11,351
  Stops GTFS disponibles para matching: 12,501
  Match exacto   (≤ 50m):  10,507  (92.6%)
  Match cercano  (≤ 100m): 641  (5.6%)
  Sin match      (> 100m): 203   (1.8%)
  Cobertura total:         11,148 (98.2%)

  Paraderos sin match confirmado (203):
              parada_bip        lat        lon  distancia_metros
20        T-28-233-NS-35 -33.502081 -70.690067        111.847903
138         L-32-17-5-PO -33.459701 -70.570383        101.624196
171       T-20-176-SN-35 -33.453851 -70.674699        105.326110
227          L-6-9-15-NS -33.335979 -70.696476        103.667395
367         L-17-28-5-NS -33.402667 -70.516306        182.903248
402        L-30-36-60-NS -33.620040 -70.681555        148.989323
430        L-16-32-50-PO -33.353812 -70.500217        168.565256
431          L-6-6-25-PO -33.324099 -70.712373        124.127998
467        L-34-29-35-PO -33.560594 -70.545328        145.1161

## Scripts 04 y 05: Limpieza de datos

### Script 04: Limpieza del dataset de Trips (2024, 2025)

In [4]:
"""
Script 04: Limpieza y tipado del dataset de Trips (2024, 2025).

Decisiones aplicadas:
  - Eliminación de 22 columnas identificadas como irrelevantes o redundantes
  - Conversión de tipos: timestamps, numéricos, categóricos
  - Valores centinela → NULL:
      '-'  en cualquier columna de texto
      -1   en comuna_fin_viaje, periodo_fin_viaje (destino desconocido)
      0    en distancia_ruta cuando distancia_eucl es NULL (destino desconocido)
  - Columnas calculadas añadidas:
      tviaje_calculado     → diferencia en segundos entre fin e inicio
      flag_destino_conocido → 1 si se conoce el destino estimado, 0 si no
      diff_dist_validacion → dveh_rutafinal + dtfinal - distancia_ruta (debe ser ~0)
  - Nulos estructurales conservados tal cual
      (etapas 2, 3, 4 ausentes son nulos válidos, no errores)

Output: data/parquets/03_clean/Trips/{anio}/trips_{anio}_clean.parquet
"""

import duckdb
from pathlib import Path

# ── Rutas ────────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
PROC_DIR  = BASE_DIR / "parquets"
RAW_DIR   = PROC_DIR / "01_raw"  # los raw parquet están en parquets/01_raw/Trips/...
CLEAN_DIR = PROC_DIR / "03_clean"


# ── Columnas a eliminar ───────────────────────────────────────────────────────
# Documentadas en analisis_de_columnas_dataset_trips_2024.txt
COLS_ELIMINAR = {
    # 100% nulas o columnas fantasma
    "tviaje", "column100", "tc3", "te3", "tv4",
    # Redundantes con tiempo_inicio_viaje
    "fecha", "mediahora_inicio_viaje_hora", "mediahora_fin_viaje_hora",
    # Mediahoras de bajada por etapa — recuperables de tiempo_bajada_k
    "mediahora_bajada_1", "mediahora_bajada_2",
    "mediahora_bajada_3", "mediahora_bajada_4",
    # Períodos de bajada por etapa — significado poco claro, redundantes
    "periodo_bajada_1", "periodo_bajada_2",
    "periodo_bajada_3", "periodo_bajada_4",
    # Operador por etapa — significado poco claro, no relevante para modelo
    "op_1era_etapa", "op_2da_etapa", "op_3era_etapa", "op_4ta_etapa",
    # Útil solo para filtrar, redundante después de eso
    "ultimaetapaconbajada",
    # Contrato de transacción — significado no documentado, irrelevante
    "contrato",
}

# ── Columnas con valor centinela -1 que deben ir a NULL ──────────────────────
COLS_MENOS1_A_NULL = {
    "comuna_fin_viaje",
    "periodo_fin_viaje",
}

# ── Columnas numéricas enteras ────────────────────────────────────────────────
COLS_INT = {
    "n_etapas", "modos", "id_viaje", "netapassinbajada",
    "periodo_inicio_viaje", "periodo_fin_viaje",
    "comuna_inicio_viaje", "comuna_fin_viaje",
    "zona_inicio_viaje", "zona_fin_viaje",
    "zona_subida_1", "zona_subida_2", "zona_subida_3", "zona_subida_4",
    "zona_bajada_1",  "zona_bajada_2",  "zona_bajada_3",  "zona_bajada_4",
}

# ── Columnas numéricas decimales ──────────────────────────────────────────────
COLS_DOUBLE = {
    # Distancias globales (metros)
    "distancia_eucl", "distancia_ruta",
    # Distancias por etapa (metros)
    "dveh_ruta1", "dveh_euc1", "dveh_ruta2", "dveh_euc2",
    "dveh_ruta3", "dveh_euc3", "dveh_ruta4", "dveh_euc4",
    # Distancias de caminata en transbordo (metros)
    "dt1", "dt2", "dt3",
    # Distancias agregadas (metros)
    "dtfinal", "dveh_rutafinal", "dveh_eucfinal",
    # Tiempos por etapa (segundos)
    "te0", "tv1", "tc1", "te1",
    "tv2", "tc2", "te2", "tv3",
    # Tiempo total (segundos)
    "tviaje2",
    # Metro: entrada y egreso (segundos)
    "entrada", "egreso",
    # Factor de expansión estadística
    "factorexpansion",
}

# ── Columnas timestamp ────────────────────────────────────────────────────────
COLS_TIMESTAMP = {
    "tiempo_inicio_viaje", "tiempo_fin_viaje",
    "tiempo_subida_1", "tiempo_subida_2", "tiempo_subida_3", "tiempo_subida_4",
    "tiempo_bajada_1", "tiempo_bajada_2", "tiempo_bajada_3", "tiempo_bajada_4",
}


# ── Helpers ───────────────────────────────────────────────────────────────────

def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def crear_conexion() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 6")
    con.execute("SET temp_directory = 'D:/temp_duckdb'")
    return con


def construir_expresion_columna(col: str) -> str:
    """
    Devuelve la expresión SQL para una columna, aplicando:
    - Conversión de '-' a NULL
    - Conversión de -1 a NULL (para columnas específicas)
    - Casteo al tipo correcto
    """
    # Expresión base: convertir '-' a NULL
    base = (
        f"CASE WHEN TRIM(CAST({col} AS VARCHAR)) = '-' "
        f"THEN NULL "
        f"ELSE {col} END"
    )

    # Aplicar conversión de -1 a NULL si corresponde
    if col in COLS_MENOS1_A_NULL:
        base = (
            f"CASE WHEN TRIM(CAST({col} AS VARCHAR)) IN ('-', '-1') "
            f"THEN NULL "
            f"ELSE {col} END"
        )

    # Aplicar casteo según tipo
    if col in COLS_INT:
        return f"TRY_CAST(({base}) AS INTEGER) AS {col}"
    elif col in COLS_DOUBLE:
        return f"TRY_CAST(({base}) AS DOUBLE) AS {col}"
    elif col in COLS_TIMESTAMP:
        return f"TRY_CAST(({base}) AS TIMESTAMP) AS {col}"
    else:
        # Columnas de texto: limpiar '-' y devolver VARCHAR
        return (
            f"CASE WHEN TRIM(CAST({col} AS VARCHAR)) = '-' "
            f"THEN NULL ELSE TRIM(CAST({col} AS VARCHAR)) END AS {col}"
        )


# ── Limpieza principal ────────────────────────────────────────────────────────

def limpiar_trips(anio: str):
    entrada = RAW_DIR / "Trips" / anio / f"trips_{anio}_raw.parquet"
    salida_dir = CLEAN_DIR / "Trips" / anio
    salida = salida_dir / f"trips_{anio}_clean.parquet"
    asegurar_carpeta(salida_dir)

    if not entrada.exists():
        print(f"  [SKIP] {entrada} no encontrado")
        return

    entrada_str = str(entrada).replace("\\", "/")
    salida_str  = str(salida).replace("\\", "/")

    print(f"\n→ Limpiando Trips {anio}...")

    con = crear_conexion()

    # Obtener columnas del archivo raw
    cols_raw = con.sql(
        f"SELECT * FROM read_parquet('{entrada_str}') LIMIT 0"
    ).df().columns.tolist()

    # Filtrar columnas a eliminar
    cols_a_usar = [c for c in cols_raw if c not in COLS_ELIMINAR]
    print(f"  Columnas raw:        {len(cols_raw)}")
    print(f"  Columnas eliminadas: {len(cols_raw) - len(cols_a_usar)}")
    print(f"  Columnas restantes:  {len(cols_a_usar)}")

    # Construir expresiones SQL para cada columna
    expresiones = [construir_expresion_columna(c) for c in cols_a_usar]

    # Columnas calculadas adicionales
    expresiones_calculadas = [
        # Tiempo de viaje calculado en segundos
        """
        TRY_CAST(
            EPOCH(
                TRY_CAST(
                    CASE WHEN TRIM(CAST(tiempo_fin_viaje AS VARCHAR)) = '-'
                    THEN NULL ELSE tiempo_fin_viaje END
                AS TIMESTAMP)
                -
                TRY_CAST(
                    CASE WHEN TRIM(CAST(tiempo_inicio_viaje AS VARCHAR)) = '-'
                    THEN NULL ELSE tiempo_inicio_viaje END
                AS TIMESTAMP)
            )
        AS DOUBLE) AS tviaje_calculado
        """,

        # Flag: ¿se conoce el destino estimado?
        """
        CASE
            WHEN TRIM(CAST(tiempo_fin_viaje AS VARCHAR)) IN ('-', '', 'NULL')
              OR tiempo_fin_viaje IS NULL
            THEN 0
            ELSE 1
        END AS flag_destino_conocido
        """,

        # Validación: diferencia entre distancia_ruta y suma de componentes
        # Debería ser cercana a 0 si los datos son consistentes
        """
        TRY_CAST(
            CASE
                WHEN TRIM(CAST(dveh_rutafinal AS VARCHAR)) = '-' THEN NULL
                WHEN TRIM(CAST(dtfinal        AS VARCHAR)) = '-' THEN NULL
                WHEN TRIM(CAST(distancia_ruta  AS VARCHAR)) = '-' THEN NULL
                ELSE TRY_CAST(dveh_rutafinal AS DOUBLE)
                   + TRY_CAST(dtfinal        AS DOUBLE)
                   - TRY_CAST(distancia_ruta  AS DOUBLE)
            END
        AS DOUBLE) AS diff_dist_validacion
        """,
    ]

    # Construir query completa
    todas_expresiones = ",\n        ".join(expresiones + expresiones_calculadas)
    query = f"""
        COPY (
            SELECT
                {todas_expresiones}
            FROM read_parquet('{entrada_str}')
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """

    con.execute(query)

    # Estadísticas del resultado
    n_raw   = con.sql(f"SELECT COUNT(*) FROM read_parquet('{entrada_str}')").fetchone()[0]
    n_clean = con.sql(f"SELECT COUNT(*) FROM read_parquet('{salida_str}')").fetchone()[0]
    n_con_destino = con.sql(
        f"SELECT COUNT(*) FROM read_parquet('{salida_str}') WHERE flag_destino_conocido = 1"
    ).fetchone()[0]
    n_sin_destino = n_clean - n_con_destino

    con.close()

    size_raw   = entrada.stat().st_size / 1e6
    size_clean = salida.stat().st_size  / 1e6

    print(f"  ✓ Completado")
    print(f"  Filas:            {n_raw:,} → {n_clean:,} (conservadas todas)")
    print(f"  Con destino:      {n_con_destino:,} ({n_con_destino/n_clean*100:.1f}%)")
    print(f"  Sin destino:      {n_sin_destino:,}  ({n_sin_destino/n_clean*100:.1f}%)")
    print(f"  Tamaño raw:       {size_raw:.1f} MB")
    print(f"  Tamaño clean:     {size_clean:.1f} MB")

In [5]:
# ── Ejecución ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 60)
    print("LIMPIEZA DE TRIPS")
    print("=" * 60)

    for anio in ["2024", "2025"]: # AGREGAR AÑOS CONFORME SEA NECESARIO
        limpiar_trips(anio)

    print("\n" + "=" * 60)
    print("✓ Limpieza de Trips completada")
    print("=" * 60)

LIMPIEZA DE TRIPS

→ Limpiando Trips 2024...
  Columnas raw:        102
  Columnas eliminadas: 22
  Columnas restantes:  80
  ✓ Completado
  Filas:            24,430,419 → 24,430,419 (conservadas todas)
  Con destino:      16,917,422 (69.2%)
  Sin destino:      7,512,997  (30.8%)
  Tamaño raw:       1727.0 MB
  Tamaño clean:     1648.7 MB

→ Limpiando Trips 2025...
  Columnas raw:        102
  Columnas eliminadas: 22
  Columnas restantes:  80
  ✓ Completado
  Filas:            21,313,043 → 21,313,043 (conservadas todas)
  Con destino:      15,276,638 (71.7%)
  Sin destino:      6,036,405  (28.3%)
  Tamaño raw:       1535.9 MB
  Tamaño clean:     1464.8 MB

✓ Limpieza de Trips completada


### Script 05: Limpieza del dataset de Stages (2025)

In [1]:
"""
Script 05: Limpieza y tipado del dataset de Stages (2025).

Decisiones aplicadas:
  - Eliminación de 4 columnas identificadas como irrelevantes o redundantes:
      fecha          → recuperable de tiempo2
      operador       → identificador de operador sin relevancia para modelo
      tiempo_subida  → redundante con tiempo2 (que cubre también zona paga)
      contrato       → codificación de contrato sin relevancia documentada

  - Renombramiento:
      id_etapa → id_tarjeta  (confirmado como ID encriptado de tarjeta bip!)

  - Conversión de tipos:
      Timestamps: tiempo2, tiempo_bajada, tiempoIniExpedicion
      Numéricos:  tiempo_etapa, x_subida, y_subida, x_bajada, y_bajada,
                  dist_ruta_paraderos, dist_eucl_paraderos,
                  tEsperaMediaIntervalo, fExpansionServicioPeriodoTS,
                  fExpansionZonaPeriodoTS, zona_subida, zona_bajada
      Enteros:    correlativo_viajes, correlativo_etapas, tiene_bajada
      Periodo:    extracción del número de periodo desde el string
                  ("05 - TRANSICION PUNTA MANANA" → 5)

  - Columna calculada añadida:
      tiempo_etapa_calculado → diferencia en segundos entre tiempo_bajada
                               y tiempo2, para validación contra tiempo_etapa

  - Nulos estructurales conservados:
      ~23% en columnas _bajada (etapas sin bajada registrada — válido)
      67.4% en tEsperaMediaIntervalo (no disponible para Metro ni ZP — válido)
      53.3% en tiempoIniExpedicion (solo presente en primera etapa — válido)

Output: data/parquets/03_clean/Stages/{anio}/stages_{anio}_clean.parquet
"""

import duckdb
from pathlib import Path

# ── Rutas ────────────────────────────────────────────────────────────────────
BASE_DIR  = Path(r"D:\GitHub\tesis_magister_route_choice_modelling\data")
PROC_DIR  = BASE_DIR / "parquets"
CLEAN_DIR = PROC_DIR / "03_clean"

# ── Columnas a eliminar ───────────────────────────────────────────────────────
COLS_ELIMINAR = {
    "fecha",          # recuperable de tiempo2
    "operador",       # identificador de operador, sin relevancia para modelo
    "tiempo_subida",  # redundante con tiempo2 (tiempo2 cubre zona paga también)
    "contrato",       # codificación de contrato, sin relevancia documentada
}

# ── Columnas numéricas decimales ──────────────────────────────────────────────
COLS_DOUBLE = {
    "tiempo_etapa",            # duración de etapa en segundos
    "x_subida", "y_subida",   # coordenadas UTM subida
    "x_bajada", "y_bajada",   # coordenadas UTM bajada (nullable)
    "dist_ruta_paraderos",     # distancia en ruta entre paraderos (nullable)
    "dist_eucl_paraderos",     # distancia euclideana entre paraderos (nullable)
    "tEsperaMediaIntervalo",   # tiempo de espera estimado en minutos (nullable)
    "fExpansionServicioPeriodoTS",  # factor de expansión por servicio y período
    "fExpansionZonaPeriodoTS",      # factor de expansión por zona y período (nullable)
}

# ── Columnas numéricas enteras ────────────────────────────────────────────────
COLS_INT = {
    "correlativo_viajes",   # número de viaje de la tarjeta en el día
    "correlativo_etapas",   # número de etapa dentro del viaje
    "tiene_bajada",         # 0/1: si la etapa tiene bajada registrada
    "zona_subida",          # zona 777 de subida
    "zona_bajada",          # zona 777 de bajada (nullable)
}

# ── Columnas timestamp ────────────────────────────────────────────────────────
COLS_TIMESTAMP = {
    "tiempo2",              # fecha/hora de subida (incluye zona paga)
    "tiempo_bajada",        # fecha/hora estimada de bajada (nullable)
    "tiempoIniExpedicion",  # fecha/hora inicio de expedición (solo 1ra etapa)
}


# ── Helpers ───────────────────────────────────────────────────────────────────

def asegurar_carpeta(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def crear_conexion() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("SET memory_limit = '3GB'")
    con.execute("SET threads = 6")
    con.execute("SET temp_directory = 'D:/temp_duckdb'")
    return con


def expresion_nulo_base(col: str) -> str:
    """Expresión SQL que convierte '-' y '' a NULL."""
    return (
        f"CASE WHEN TRIM(CAST({col} AS VARCHAR)) IN ('-', '') "
        f"THEN NULL ELSE {col} END"
    )


def construir_expresion(col: str, alias: str = None) -> str:
    """
    Construye la expresión SQL para una columna con su tipo correcto.
    alias permite renombrar la columna en el output.
    """
    nombre_salida = alias if alias else col
    base = expresion_nulo_base(col)

    if col in COLS_DOUBLE:
        return f"TRY_CAST(({base}) AS DOUBLE) AS {nombre_salida}"
    elif col in COLS_INT:
        return f"TRY_CAST(({base}) AS INTEGER) AS {nombre_salida}"
    elif col in COLS_TIMESTAMP:
        return f"TRY_CAST(({base}) AS TIMESTAMP) AS {nombre_salida}"
    else:
        return (
            f"CASE WHEN TRIM(CAST({col} AS VARCHAR)) IN ('-', '') "
            f"THEN NULL ELSE TRIM(CAST({col} AS VARCHAR)) END AS {nombre_salida}"
        )


# ── Limpieza principal ────────────────────────────────────────────────────────

def limpiar_stages(anio: str):
    entrada = PROC_DIR / "01_raw" / "Stages" / anio / f"stages_{anio}_raw.parquet"
    salida_dir = CLEAN_DIR / "Stages" / anio
    salida = salida_dir / f"stages_{anio}_clean.parquet"
    asegurar_carpeta(salida_dir)

    if not entrada.exists():
        print(f"  [SKIP] {entrada} no encontrado")
        return

    entrada_str = str(entrada).replace("\\", "/")
    salida_str  = str(salida).replace("\\", "/")

    print(f"\n→ Limpiando Stages {anio}...")

    con = crear_conexion()

    cols_raw = con.sql(
        f"SELECT * FROM read_parquet('{entrada_str}') LIMIT 0"
    ).df().columns.tolist()

    cols_a_usar = [c for c in cols_raw if c not in COLS_ELIMINAR]
    print(f"  Columnas raw:        {len(cols_raw)}")
    print(f"  Columnas eliminadas: {len(cols_raw) - len(cols_a_usar)}")
    print(f"  Columnas restantes:  {len(cols_a_usar)}")

    # Construir expresiones con renombramiento de id_etapa → id_tarjeta
    expresiones = []
    for col in cols_a_usar:
        alias = "id_tarjeta" if col == "id_etapa" else None
        expresiones.append(construir_expresion(col, alias))

    # Columna calculada: extracción del número de periodo desde el string
    # "05 - TRANSICION PUNTA MANANA" → 5
    expresiones_calculadas = [
        """
        TRY_CAST(
            CASE
                WHEN periodoSubida IS NULL
                  OR TRIM(CAST(periodoSubida AS VARCHAR)) IN ('-', '')
                THEN NULL
                ELSE SPLIT_PART(TRIM(CAST(periodoSubida AS VARCHAR)), ' ', 1)
            END
        AS INTEGER) AS periodo_subida_num
        """,

        """
        TRY_CAST(
            CASE
                WHEN periodoBajada IS NULL
                  OR TRIM(CAST(periodoBajada AS VARCHAR)) IN ('-', '')
                THEN NULL
                ELSE SPLIT_PART(TRIM(CAST(periodoBajada AS VARCHAR)), ' ', 1)
            END
        AS INTEGER) AS periodo_bajada_num
        """,

        # Validación: diferencia calculada vs tiempo_etapa declarado
        # Debería ser ~0 si los datos son consistentes
        """
        TRY_CAST(
            CASE
                WHEN tiempo_bajada IS NULL
                  OR TRIM(CAST(tiempo_bajada AS VARCHAR)) IN ('-', '')
                  OR tiempo2 IS NULL
                  OR TRIM(CAST(tiempo2 AS VARCHAR)) IN ('-', '')
                THEN NULL
                ELSE EPOCH(
                    TRY_CAST(tiempo_bajada AS TIMESTAMP) -
                    TRY_CAST(tiempo2 AS TIMESTAMP)
                )
            END
        AS DOUBLE) AS tiempo_etapa_calculado
        """,
    ]

    todas_expresiones = ",\n        ".join(expresiones + expresiones_calculadas)

    query = f"""
        COPY (
            SELECT
                {todas_expresiones}
            FROM read_parquet('{entrada_str}')
        )
        TO '{salida_str}'
        (FORMAT PARQUET, COMPRESSION 'ZSTD', ROW_GROUP_SIZE 100000)
    """

    con.execute(query)

    # Estadísticas del resultado
    n_raw   = con.sql(
        f"SELECT COUNT(*) FROM read_parquet('{entrada_str}')"
    ).fetchone()[0]

    n_clean = con.sql(
        f"SELECT COUNT(*) FROM read_parquet('{salida_str}')"
    ).fetchone()[0]

    n_con_bajada = con.sql(f"""
        SELECT COUNT(*)
        FROM read_parquet('{salida_str}')
        WHERE tiene_bajada = 1
    """).fetchone()[0]

    n_sin_bajada = n_clean - n_con_bajada

    # Validación de consistencia tiempo_etapa
    diff_stats = con.sql(f"""
        SELECT
            AVG(ABS(tiempo_etapa_calculado - tiempo_etapa)) AS diff_media,
            MAX(ABS(tiempo_etapa_calculado - tiempo_etapa)) AS diff_max,
            COUNT(*) FILTER (
                WHERE tiempo_etapa_calculado IS NOT NULL
                  AND tiempo_etapa IS NOT NULL
            ) AS n_comparables
        FROM read_parquet('{salida_str}')
    """).fetchone()

    con.close()

    size_raw   = entrada.stat().st_size / 1e6
    size_clean = salida.stat().st_size  / 1e6

    print(f"  ✓ Completado")
    print(f"  Filas:              {n_raw:,} → {n_clean:,}")
    print(f"  Con bajada:         {n_con_bajada:,} ({n_con_bajada/n_clean*100:.1f}%)")
    print(f"  Sin bajada:         {n_sin_bajada:,}  ({n_sin_bajada/n_clean*100:.1f}%)")
    print(f"  Columnas finales:   {len(cols_a_usar) + 3}")
    print(f"  Tamaño raw:         {size_raw:.1f} MB")
    print(f"  Tamaño clean:       {size_clean:.1f} MB")

    if diff_stats[2] and diff_stats[2] > 0:
        print(f"\n  Validación tiempo_etapa vs calculado:")
        print(f"    Diferencia media: {diff_stats[0]:.2f} seg")
        print(f"    Diferencia máx:   {diff_stats[1]:.2f} seg")
        print(f"    Filas comparadas: {diff_stats[2]:,}")
        if diff_stats[0] < 60:
            print(f"    ✓ Consistencia aceptable (diff media < 60 seg)")
        else:
            print(f"    ⚠ Diferencias elevadas — revisar tiempo2 vs tiempo_subida")

In [2]:
# ── Ejecución ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 60)
    print("LIMPIEZA DE STAGES")
    print("=" * 60)

    for anio in ["2025"]: # AGREGAR AÑOS CONFORME SEA NECESARIO
        limpiar_stages(anio)

    print("\n" + "=" * 60)
    print("✓ Limpieza de Stages completada")
    print("=" * 60)

LIMPIEZA DE STAGES

→ Limpiando Stages 2025...
  Columnas raw:        36
  Columnas eliminadas: 4
  Columnas restantes:  32
  ✓ Completado
  Filas:              27,949,622 → 27,949,622
  Con bajada:         21,493,466 (76.9%)
  Sin bajada:         6,456,156  (23.1%)
  Columnas finales:   35
  Tamaño raw:         1340.6 MB
  Tamaño clean:       1273.1 MB

  Validación tiempo_etapa vs calculado:
    Diferencia media: 0.00 seg
    Diferencia máx:   0.00 seg
    Filas comparadas: 21,493,466
    ✓ Consistencia aceptable (diff media < 60 seg)

✓ Limpieza de Stages completada
